# Casys SDK — Variable-Length Paths (ISO GQL)

Ce notebook valide la syntaxe `*min..max` pour les chemins à longueur variable,
via des requêtes GQL directes.

In [1]:
from casys_db import CasysEngine
import tempfile

# Préparation d'un espace de travail temporaire
data_dir = tempfile.mkdtemp()
engine = CasysEngine(data_dir)
db = engine.open_database("social")
branch = engine.open_branch("social", "main")

# Création d'un petit graphe: Alice -> Bob -> Charlie -> David
a = branch.add_node(["Person"], {"name": "Alice"})
b = branch.add_node(["Person"], {"name": "Bob"})
c = branch.add_node(["Person"], {"name": "Charlie"})
d = branch.add_node(["Person"], {"name": "David"})
branch.add_edge(a, b, "KNOWS", {})
branch.add_edge(b, c, "KNOWS", {})
branch.add_edge(c, d, "KNOWS", {})

print(f'Data dir: {data_dir}')

Data dir: /tmp/tmp_kf6f9ck


## GQL — Cas d'usage: exact, plage, borne basse/haute

In [2]:
# Exactement 2 sauts: *2
res_exact_2 = branch.query("MATCH (a:Person)-[:KNOWS*2]->(p:Person) WHERE a.name = 'Alice' RETURN p.name")
print('Exact 2 hops →', res_exact_2)

# 1 à 3 sauts: *1..3
res_1_3 = branch.query("MATCH (a:Person)-[:KNOWS*1..3]->(p:Person) WHERE a.name = 'Alice' RETURN p.name")
print('1..3 hops →', res_1_3)

# 0 à 2 sauts: *..2 (inclut le nœud de départ)
res_0_2 = branch.query("MATCH (a:Person)-[:KNOWS*..2]->(p:Person) WHERE a.name = 'Alice' RETURN p.name")
print('0..2 hops →', res_0_2)

# Au moins 2 sauts: *2..
res_min_2 = branch.query("MATCH (a:Person)-[:KNOWS*2..]->(p:Person) WHERE a.name = 'Alice' RETURN p.name")
print('2..∞ hops →', res_min_2)

# Tout: *
res_any = branch.query("MATCH (a:Person)-[:KNOWS*]->(p:Person) WHERE a.name = 'Alice' RETURN p.name")
print('1..∞ hops →', res_any)

Exact 2 hops → {'columns': ['p.name'], 'rows': [['Charlie']]}
1..3 hops → {'columns': ['p.name'], 'rows': [['Bob'], ['Charlie'], ['David']]}
0..2 hops → {'columns': ['p.name'], 'rows': [['Bob'], ['Charlie']]}
2..∞ hops → {'columns': ['p.name'], 'rows': [['Charlie'], ['David']]}
1..∞ hops → {'columns': ['p.name'], 'rows': [['Bob'], ['Charlie'], ['David']]}


## ORM — Rappel API (à titre indicatif)

L'ORM propose `depth()` pour configurer la longueur des traversées lors de la navigation
(conversion interne en GQL):

- `depth(n)` → `*n` (exact n)
- `depth(n, m)` → `*n..m`
- `depth(max=m)` → `*1..m`
- `depth(min=n)` → `*n..`
- `depth()` → `*`
- `depth(0, m)` → `*0..m`